# w20_gpu1_decoupled_m3_and_true20f_mono
Inputs: `shortpose-wave1-private-inputs` | Accelerator: `GPU T4 x2`


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
import torch

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir() and torch.cuda.is_available(), 'Kaggle GPU required'
n_gpu = torch.cuda.device_count()
REV = '428732b1a3f1ee39efe893287dcd2fdb56fe010a'
with tempfile.TemporaryDirectory(prefix='w20_g1_') as ts:
    tmp = Path(ts)
    repo, meta_dir, sc_dir = tmp / 'repo', tmp / 'pinns_meta', tmp / 'scratch'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    base_env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1'}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    map_bin = next(MOUNT.rglob('pinns_source_label_causal_maps.bin'))
    matched_npz = next(MOUNT.rglob('pinns_matched_histories.npz'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    obs_hits = list(MOUNT.rglob('matched_observations.npz'))
    if obs_hits:
        matched_obs = obs_hits[0]
    else:
        md_zips = list(MOUNT.rglob('matched_durations.zip'))
        matched_obs = tmp / 'matched_observations.npz'
        if md_zips:
            with zipfile.ZipFile(md_zips[0]) as zf:
                nm = next(n for n in zf.namelist() if n.endswith('matched_observations.npz'))
                matched_obs.write_bytes(zf.read(nm))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=base_env, check=True)
    d0 = WORK / 'wave20_decoupled_m2b_m3_synergy_5seed.zip'
    d1 = WORK / 'wave20_true20f_mono_3x3_matrix_5seed.zip'
    e0 = {**base_env, 'CUDA_VISIBLE_DEVICES': '0'}
    e1 = {**base_env, 'CUDA_VISIBLE_DEVICES': '1' if n_gpu > 1 else '0'}
    p0 = subprocess.Popen([
        sys.executable, '-u', '-m', 'scripts.run_wave20_decoupled_m2b_m3_synergy_5seed',
        '--repo-root', str(repo), '--meta-dir', str(meta_dir), '--pose-npz', str(pose_npz),
        '--joint-zip', str(joint_bin), '--causal-map-bin', str(map_bin), '--zip-path', str(d0)
    ], cwd=repo, env=e0)
    p1 = subprocess.Popen([
        sys.executable, '-u', '-m', 'scripts.run_wave20_true20f_mono_3x3_matrix_5seed',
        '--matched-npz', str(matched_npz), '--meta-dir', str(meta_dir), '--pose-npz', str(pose_npz),
        '--matched-obs-npz', str(matched_obs), '--scratch-dir', str(sc_dir), '--zip-path', str(d1)
    ], cwd=repo, env=e1)
    rc0, rc1 = p0.wait(), p1.wait()
    assert rc0 == 0 and rc1 == 0, f'Subprocess failed: rc0={rc0}, rc1={rc1}'
    for d in (d0, d1):
        print('DONE:', d, d.stat().st_size, hashlib.sha256(d.read_bytes()).hexdigest(), flush=True)
